# Notebook 04: `Voxel51/WLASL` Top-100 Multimodal Vision-Language Training

**Runtime:** Works fast on **Colab CPU (`~4–8 min` total)** or **T4 GPU (`< 1 min`)**.  
**Strict Split Discipline:** Trains ONLY on `train` (`672` clips, `31` signers) and selects/compares models strictly on `val` (`218` clips, `17` unseen signers). **`test` (`230` clips) remains locked until Notebook 05.**

### How This Notebook Addresses Notebook 03 Overfitting
1. **Smaller bottleneck (`hidden=128`, `embed_dim=128`)** to cut parameter count vs. `672` training samples.
2. **Stronger regularization:** `weight_decay=1e-2`, `dropout=0.45`, channel dropout (`0.25`), temporal cutout (`2–4` contiguous frames masked), Gaussian noise (`0.03`), and **Feature Mixup (`α=0.4`)** during training.
3. **Bounded unit-hypersphere alignment:** `L2`-normalized visual and language embeddings (`||v|| = ||t|| = 1`) with temperature clamping + **multi-positive symmetric InfoNCE** + **cross-modal gated fusion**.
4. **Warmup + Cosine LR** (`5`-epoch warmup) to avoid early optimization spikes.

### Experiments Benchmarked (`val` split)
1. `visual_transformer_reg` — Regularized Visual v2 (`Transformer(128) + Mixup/Cutout/WD=1e-2`)
2. `mm_contrastive_gloss` — Multimodal (`contrastive`, raw gloss embeddings)
3. `mm_contrastive_prompt` — Multimodal (`contrastive`, ASL prompt embeddings)
4. `mm_hybrid_transformer` — Multimodal (`hybrid`, best text variant)
5. `mm_gated_fusion_transformer` — Multimodal (`gated_fusion`, cross-modal attention + sigmoid gate)
6. `mm_gated_fusion_bilstm` — Multimodal (`gated_fusion` paired with `BiLSTM(128)`)

All saved `best_model.pt` checkpoints include `vocab_text_embeddings` and match `src/inference/predictor.py` (`strict=True`).


## 1. Environment Setup & Google Drive Mount


In [ ]:
import json
import math
import random
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from google.colab import drive as colab_drive
    IN_COLAB = True
except ImportError:
    colab_drive = None
    IN_COLAB = False

if IN_COLAB and colab_drive is not None:
    if Path("/content/drive/MyDrive").exists():
        print("Google Drive already mounted at /content/drive.")
    else:
        colab_drive.mount("/content/drive")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Device: {DEVICE} | IN_COLAB={IN_COLAB}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

## 2. Load Cached Visual & Language Features (`train` / `val` Only)


In [ ]:
# ==============================================================================
# CONFIGURATION & DATA LOADING (TRAIN / VAL ONLY)
# ==============================================================================
SEED = 42
NUM_CLASSES = 100
FEATURE_DIM = 2048
BATCH_SIZE = 32
EPOCHS = 80
WARMUP_EPOCHS = 5
PATIENCE = 22


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

if IN_COLAB and Path("/content/drive/MyDrive").exists():
    DRIVE_ROOT = Path("/content/drive/MyDrive/ASL-Recognition/voxel51_wlasl100")
else:
    DRIVE_ROOT = Path.cwd() / "data" / "voxel51_wlasl100"

META_DIR = DRIVE_ROOT / "metadata"
FEAT_DIR = DRIVE_ROOT / "features"
CKPT_DIR = DRIVE_ROOT / "checkpoints"
EXP_DIR = DRIVE_ROOT / "experiments"
PLOTS_DIR = DRIVE_ROOT / "reports" / "plots"

for d in (CKPT_DIR, EXP_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

class_to_id = {str(k): int(v) for k, v in json.loads((META_DIR / "class_to_id.json").read_text(encoding="utf-8")).items()}
id_to_class = {int(k): str(v) for k, v in json.loads((META_DIR / "id_to_class.json").read_text(encoding="utf-8")).items()}
assert len(class_to_id) == NUM_CLASSES


def load_split(sp: str):
    x = np.asarray(np.load(FEAT_DIR / f"X_{sp}.npy"), dtype=np.float32)
    y = np.asarray(np.load(FEAT_DIR / f"y_{sp}.npy"), dtype=np.int64)
    done = np.load(FEAT_DIR / f"done_{sp}.npy")
    assert bool(done.all()) and np.isfinite(x).all()
    return x, y


X_train, y_train = load_split("train")
X_val, y_val = load_split("val")

emb_gloss_obj = torch.load(FEAT_DIR / "text_embeddings_gloss.pt", map_location="cpu", weights_only=False)
emb_prompt_obj = torch.load(FEAT_DIR / "text_embeddings_prompt.pt", map_location="cpu", weights_only=False)
vocab_gloss = emb_gloss_obj["text_embeddings"].float().to(DEVICE)
vocab_prompt = emb_prompt_obj["text_embeddings"].float().to(DEVICE)

print(f"Train: {X_train.shape} | Val: {X_val.shape} (Test kept locked)")
print(f"Vocab embeddings: gloss={tuple(vocab_gloss.shape)}, prompt={tuple(vocab_prompt.shape)}")

## 3. Strong Augmentations (`Cutout`, `Mixup`) & Evaluation Metrics


In [ ]:
# ==============================================================================
# AUGMENTATIONS (TEMPORAL CROP, CUTOUT, MIXUP) & CLASSIFICATION/RETRIEVAL METRICS
# ==============================================================================
def augment_feature_sequence_strong(
    features: np.ndarray,
    *,
    crop_ratio_range: tuple[float, float] = (0.80, 1.0),
    feature_dropout: float = 0.25,
    noise_std: float = 0.03,
    cutout_max_frames: int = 4,
    rng: np.random.Generator | None = None,
) -> np.ndarray:
    gen = rng if rng is not None else np.random.default_rng()
    total_len = features.shape[0]
    ratio = float(gen.uniform(crop_ratio_range[0], crop_ratio_range[1]))
    win_len = max(1, min(total_len, int(round(total_len * ratio))))
    max_start = total_len - win_len
    start = int(gen.integers(0, max_start + 1)) if max_start > 0 else 0
    end = start + win_len - 1
    indices = np.round(np.linspace(start, end, total_len)).astype(np.int64)
    aug = features[indices].astype(np.float32, copy=True)

    # Random contiguous temporal cutout (prevents memorizing 1 lucky frame)
    if cutout_max_frames > 0 and total_len > cutout_max_frames and gen.random() < 0.5:
        cut_len = int(gen.integers(1, cutout_max_frames + 1))
        cut_start = int(gen.integers(0, total_len - cut_len + 1))
        aug[cut_start : cut_start + cut_len] = 0.0

    # Structured channel-wise + elementwise feature dropout
    if feature_dropout > 0.0:
        keep_prob = 1.0 - float(feature_dropout)
        chan_mask = (gen.random((1, aug.shape[1])) < keep_prob).astype(np.float32)
        aug = (aug * chan_mask) / max(keep_prob, 1e-6)

    if noise_std > 0.0:
        aug = aug + gen.normal(0.0, float(noise_std), size=aug.shape).astype(np.float32)
    return aug


class CachedFeatureDataset(Dataset):
    def __init__(self, features: np.ndarray, labels: np.ndarray, augment: bool = False):
        self.features = features
        self.labels = labels
        self.augment = augment

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int):
        seq = np.array(self.features[idx], dtype=np.float32, copy=True)
        if self.augment:
            seq = augment_feature_sequence_strong(seq)
        return torch.from_numpy(seq), torch.tensor(int(self.labels[idx]), dtype=torch.long)


def make_balanced_sampler(labels: np.ndarray, power: float = 0.5) -> WeightedRandomSampler:
    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(np.float64)
    cls_w = np.where(counts > 0, 1.0 / np.power(np.maximum(counts, 1.0), power), 0.0)
    return WeightedRandomSampler(weights=cls_w[labels].tolist(), num_samples=len(labels), replacement=True)


def compute_classification_metrics(y_true: np.ndarray, logits: np.ndarray) -> dict[str, Any]:
    y_t = np.asarray(y_true, dtype=np.int64)
    scores = np.asarray(logits, dtype=np.float64)
    y_pred = np.argmax(scores, axis=1)
    top1 = float(np.mean(y_pred == y_t))
    top5_idx = np.argpartition(-scores, kth=min(4, scores.shape[1] - 1), axis=1)[:, :5]
    top5 = float(np.mean(np.any(top5_idx == y_t[:, None], axis=1)))
    f1s = []
    for cid in range(scores.shape[1]):
        tp = int(np.sum((y_t == cid) & (y_pred == cid)))
        fp = int(np.sum((y_t != cid) & (y_pred == cid)))
        fn = int(np.sum((y_t == cid) & (y_pred != cid)))
        if tp + fn == 0:
            continue
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn)
        f1s.append((2.0 * p * r / (p + r)) if (p + r) > 0 else 0.0)
    return {
        "samples": int(len(y_t)),
        "top1_accuracy": top1,
        "top5_accuracy": top5,
        "macro_f1": float(np.mean(f1s)),
        "unique_predicted_classes": int(len(np.unique(y_pred))),
    }


def compute_retrieval_metrics(sim_matrix: np.ndarray, q_labels: np.ndarray, g_labels: np.ndarray) -> dict[str, float]:
    ranked = np.argsort(-sim_matrix, axis=1)
    ranks = []
    for i, order in enumerate(ranked):
        hits = np.flatnonzero(g_labels[order] == q_labels[i])
        ranks.append(int(hits[0]) + 1)
    ranks_arr = np.asarray(ranks, dtype=np.float64)
    return {
        "mrr": float(np.mean(1.0 / ranks_arr)),
        "median_rank": float(np.median(ranks_arr)),
        "recall@1": float(np.mean(ranks_arr <= 1)),
        "recall@5": float(np.mean(ranks_arr <= 5)),
        "recall@10": float(np.mean(ranks_arr <= 10)),
    }

## 4. Multimodal Vision-Language & Regularized Visual Architectures


In [ ]:
# ==============================================================================
# MULTIMODAL & VISUAL ARCHITECTURES (EXACT src/ STATE_DICT MATCH)
# ==============================================================================
class TemporalAttentionPool(nn.Module):
    def __init__(self, dim: int) -> None:
        super().__init__()
        self.proj = nn.Linear(dim, 1)

    def forward(self, sequence: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(self.proj(sequence), dim=1)
        return (sequence * weights).sum(dim=1)


class TemporalLSTM(nn.Module):
    def __init__(self, input_dim=2048, hidden_dim=128, num_layers=2, dropout=0.45, bidirectional=True, pooling="attention"):
        super().__init__()
        self.input_proj = nn.Sequential(nn.LayerNorm(input_dim), nn.Dropout(dropout))
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        self.output_dim = hidden_dim * (2 if bidirectional else 1)
        self.pooling = pooling
        self.attn_pool = TemporalAttentionPool(self.output_dim) if pooling == "attention" else None

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(features)
        sequence, (hidden, _) = self.lstm(x)
        if self.pooling == "mean":
            return sequence.mean(dim=1)
        if self.pooling == "attention" and self.attn_pool is not None:
            return self.attn_pool(sequence)
        if self.lstm.bidirectional:
            return torch.cat([hidden[-2], hidden[-1]], dim=-1)
        return hidden[-1]


class TemporalTransformerEncoder(nn.Module):
    def __init__(
        self,
        input_dim: int = 2048,
        embed_dim: int = 128,
        num_layers: int = 2,
        num_heads: int = 4,
        dim_feedforward: int = 512,
        dropout: float = 0.45,
        max_seq_len: int = 128,
        pooling: str = "attention",
    ) -> None:
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        pe = torch.zeros(1, max_seq_len, embed_dim)
        pos = torch.arange(0, max_seq_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, embed_dim, 2, dtype=torch.float32) * (-math.log(10000.0) / embed_dim))
        pe[0, :, 0::2] = torch.sin(pos * div)
        pe[0, :, 1::2] = torch.cos(pos * div)
        self.register_buffer("pos_encoding", pe, persistent=False)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(embed_dim)
        self.output_dim = embed_dim
        self.pooling = pooling
        self.attn_pool = TemporalAttentionPool(embed_dim) if pooling == "attention" else None

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        t_len = features.size(1)
        x = self.input_proj(features) + self.pos_encoding[:, :t_len, :]
        encoded = self.norm(self.encoder(x))
        if self.pooling == "attention" and self.attn_pool is not None:
            return self.attn_pool(encoded)
        return encoded.mean(dim=1)


class VisualClassifier(nn.Module):
    def __init__(self, num_classes=100, feature_dim=2048, hidden_dim=128, num_layers=2, dropout=0.45, temporal_type="transformer", bidirectional=False, pooling="attention"):
        super().__init__()
        if temporal_type == "lstm":
            self.temporal = TemporalLSTM(feature_dim, hidden_dim, num_layers, dropout, bidirectional, pooling)
        else:
            self.temporal = TemporalTransformerEncoder(feature_dim, hidden_dim, num_layers, 4, hidden_dim * 2, dropout, pooling="mean" if pooling == "last" else pooling)
        self.head_dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.temporal.output_dim, num_classes)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.head_dropout(self.temporal(inputs)))


class VisualProjectionHead(nn.Module):
    def __init__(self, input_dim: int, embed_dim: int = 128, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.net(x), p=2, dim=-1)


class TextProjectionHead(nn.Module):
    def __init__(self, input_dim: int = 384, embed_dim: int = 128, dropout: float = 0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.net(x), p=2, dim=-1)


class MultimodalASLModel(nn.Module):
    def __init__(
        self,
        num_classes: int = 100,
        *,
        visual_feature_dim: int = 2048,
        temporal_hidden_dim: int = 128,
        temporal_layers: int = 2,
        temporal_type: str = "transformer",
        bidirectional: bool = False,
        pooling: str = "attention",
        text_dim: int = 384,
        embed_dim: int = 128,
        dropout: float = 0.45,
        fusion_mode: str = "gated_fusion",
        fusion_alpha: float = 0.5,
        init_temperature: float = 0.1,
    ) -> None:
        super().__init__()
        self.num_classes = num_classes
        self.fusion_mode = fusion_mode
        self.fusion_alpha = float(fusion_alpha)
        if temporal_type == "lstm":
            self.temporal_encoder = TemporalLSTM(visual_feature_dim, temporal_hidden_dim, temporal_layers, dropout, bidirectional, pooling)
        else:
            # Matches MultimodalASLModel default dim_feedforward=512 in src/
            self.temporal_encoder = TemporalTransformerEncoder(visual_feature_dim, temporal_hidden_dim, temporal_layers, 4, 512, dropout, pooling="mean" if pooling == "last" else pooling)
        repr_dim = self.temporal_encoder.output_dim
        self.visual_projection = VisualProjectionHead(repr_dim, embed_dim, dropout=dropout)
        self.text_projection = TextProjectionHead(text_dim, embed_dim, dropout=0.15)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1.0 / init_temperature), dtype=torch.float32))
        self.visual_classifier = nn.Linear(repr_dim, num_classes)
        if fusion_mode == "gated_fusion":
            self.fusion_gate = nn.Sequential(nn.Linear(embed_dim * 2, embed_dim), nn.Sigmoid())
            self.fused_classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(embed_dim, num_classes))
        else:
            self.fusion_gate = None
            self.fused_classifier = None

    def forward(self, visual_inputs: torch.Tensor, vocab_text_embeddings: torch.Tensor) -> dict[str, torch.Tensor]:
        repr_vec = self.temporal_encoder(visual_inputs)
        visual_emb = self.visual_projection(repr_vec)
        text_emb = self.text_projection(vocab_text_embeddings)
        scale = self.logit_scale.exp().clamp(max=30.0)
        cosine_logits = scale * (visual_emb @ text_emb.T)

        if self.fusion_mode == "contrastive":
            cls_logits = cosine_logits
        elif self.fusion_mode == "hybrid":
            cls_logits = self.fusion_alpha * cosine_logits + (1.0 - self.fusion_alpha) * self.visual_classifier(repr_vec)
        else:
            attn = torch.softmax(cosine_logits, dim=-1)
            lang_ctx = attn @ text_emb
            gate = self.fusion_gate(torch.cat([visual_emb, lang_ctx], dim=-1))
            fused = gate * visual_emb + (1.0 - gate) * lang_ctx
            cls_logits = self.fused_classifier(fused) + self.fusion_alpha * cosine_logits

        return {
            "logits": cls_logits,
            "cosine_logits": cosine_logits,
            "visual_embeddings": visual_emb,
            "text_embeddings": text_emb,
            "logit_scale": scale,
        }


def multi_positive_infonce(visual_emb: torch.Tensor, text_emb: torch.Tensor, logit_scale: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    logits_v2t = logit_scale * (visual_emb @ text_emb.T)
    same_cls = (labels.unsqueeze(1) == labels.unsqueeze(0)).float()
    targets = same_cls / same_cls.sum(dim=1, keepdim=True).clamp(min=1.0)
    loss_v = -(targets * F.log_softmax(logits_v2t, dim=1)).sum(dim=1).mean()
    loss_t = -(targets.T * F.log_softmax(logits_v2t.T, dim=1)).sum(dim=1).mean()
    return 0.5 * (loss_v + loss_t)

## 5. Training Engine with Warmup, Mixup & Multi-Positive InfoNCE


In [ ]:
# ==============================================================================
# TRAINING ENGINE (WITH WARMUP, MIXUP, WEIGHT DECAY & RETRIEVAL LOGGING)
# ==============================================================================
def make_warmup_cosine_scheduler(optimizer: torch.optim.Optimizer, epochs: int, warmup_epochs: int = WARMUP_EPOCHS):
    def lr_lambda(ep: int) -> float:
        if ep < warmup_epochs:
            return float(ep + 1) / float(max(1, warmup_epochs))
        progress = float(ep - warmup_epochs) / float(max(1, epochs - warmup_epochs))
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def apply_mixup(xb: torch.Tensor, yb: torch.Tensor, alpha: float = 0.4):
    if alpha <= 0.0 or xb.size(0) < 2:
        return xb, yb, yb, 1.0
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(xb.size(0), device=xb.device)
    mixed_x = lam * xb + (1.0 - lam) * xb[perm]
    return mixed_x, yb, yb[perm], lam


def train_and_eval_run(
    exp_id: str,
    model: nn.Module,
    *,
    is_multimodal: bool,
    vocab_tensor: torch.Tensor | None = None,
    model_cfg: dict[str, Any],
    lr: float = 7e-4,
    weight_decay: float = 1e-2,
    label_smoothing: float = 0.10,
    mixup_alpha: float = 0.4,
    contrastive_weight: float = 0.5,
    classification_weight: float = 0.5,
    batch_infonce_weight: float = 0.25,
) -> dict[str, Any]:
    set_seed(SEED)
    model = model.to(DEVICE)
    run_dir = CKPT_DIR / exp_id
    run_dir.mkdir(parents=True, exist_ok=True)

    sampler = make_balanced_sampler(y_train, power=0.5)
    tr_loader = DataLoader(CachedFeatureDataset(X_train, y_train, augment=True), batch_size=BATCH_SIZE, sampler=sampler)
    va_loader = DataLoader(CachedFeatureDataset(X_val, y_val, augment=False), batch_size=BATCH_SIZE, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = make_warmup_cosine_scheduler(optimizer, EPOCHS, WARMUP_EPOCHS)

    best_score = (-1.0, float("-inf"))
    best_epoch = 0
    best_metrics: dict[str, Any] = {}
    no_improve = 0
    history = []

    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        tr_loss_sum, tr_n = 0.0, 0
        for xb, yb in tr_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            mx, y_a, y_b, lam = apply_mixup(xb, yb, alpha=mixup_alpha)
            optimizer.zero_grad(set_to_none=True)

            if is_multimodal:
                assert vocab_tensor is not None
                out = model(mx, vocab_tensor)
                ce_a = F.cross_entropy(out["logits"], y_a, label_smoothing=label_smoothing)
                ce_b = F.cross_entropy(out["logits"], y_b, label_smoothing=label_smoothing)
                cls_loss = lam * ce_a + (1.0 - lam) * ce_b

                vc_a = F.cross_entropy(out["cosine_logits"], y_a, label_smoothing=label_smoothing)
                vc_b = F.cross_entropy(out["cosine_logits"], y_b, label_smoothing=label_smoothing)
                vocab_loss = lam * vc_a + (1.0 - lam) * vc_b

                batch_txt_a = out["text_embeddings"][y_a]
                batch_txt_b = out["text_embeddings"][y_b]
                inf_a = multi_positive_infonce(out["visual_embeddings"], batch_txt_a, out["logit_scale"], y_a)
                inf_b = multi_positive_infonce(out["visual_embeddings"], batch_txt_b, out["logit_scale"], y_b)
                inf_loss = lam * inf_a + (1.0 - lam) * inf_b

                loss = classification_weight * cls_loss + contrastive_weight * vocab_loss + batch_infonce_weight * inf_loss
            else:
                logits = model(mx)
                loss = lam * F.cross_entropy(logits, y_a, label_smoothing=label_smoothing) + (1.0 - lam) * F.cross_entropy(logits, y_b, label_smoothing=label_smoothing)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            bs = yb.size(0)
            tr_n += bs
            tr_loss_sum += float(loss.detach()) * bs

        scheduler.step()

        # Validation evaluation
        model.eval()
        val_loss_sum, val_n = 0.0, 0
        all_logits, all_vis, all_labels = [], [], []
        txt_proto = None
        with torch.inference_mode():
            for xb, yb in va_loader:
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                if is_multimodal:
                    assert vocab_tensor is not None
                    out = model(xb, vocab_tensor)
                    loss = F.cross_entropy(out["logits"], yb, label_smoothing=label_smoothing)
                    all_logits.append(out["logits"].cpu().numpy())
                    all_vis.append(out["visual_embeddings"].cpu().numpy())
                    txt_proto = out["text_embeddings"].cpu().numpy()
                else:
                    logits = model(xb)
                    loss = F.cross_entropy(logits, yb, label_smoothing=label_smoothing)
                    all_logits.append(logits.cpu().numpy())
                all_labels.append(yb.cpu().numpy())
                val_n += yb.size(0)
                val_loss_sum += float(loss.detach()) * yb.size(0)

        logits_np = np.concatenate(all_logits, axis=0)
        labels_np = np.concatenate(all_labels, axis=0)
        val_m = compute_classification_metrics(labels_np, logits_np)
        val_m["loss"] = val_loss_sum / max(1, val_n)

        if is_multimodal and txt_proto is not None:
            vis_np = np.concatenate(all_vis, axis=0)
            sim_v2t = vis_np @ txt_proto.T
            ret_v2t = compute_retrieval_metrics(sim_v2t, labels_np, np.arange(NUM_CLASSES))
            ret_t2v = compute_retrieval_metrics(sim_v2t.T, np.arange(NUM_CLASSES), labels_np)
            val_m["v2t_recall@1"] = ret_v2t["recall@1"]
            val_m["v2t_recall@5"] = ret_v2t["recall@5"]
            val_m["v2t_recall@10"] = ret_v2t["recall@10"]
            val_m["v2t_mrr"] = ret_v2t["mrr"]
            val_m["t2v_recall@1"] = ret_t2v["recall@1"]
            val_m["t2v_recall@5"] = ret_t2v["recall@5"]

        row = {
            "epoch": epoch,
            "train_loss": tr_loss_sum / max(1, tr_n),
            "val_loss": val_m["loss"],
            "val_top1": val_m["top1_accuracy"],
            "val_top5": val_m["top5_accuracy"],
            "val_macro_f1": val_m["macro_f1"],
            "v2t_r5": val_m.get("v2t_recall@5", np.nan),
            "unique_preds": val_m["unique_predicted_classes"],
        }
        history.append(row)

        cur_score = (val_m["top1_accuracy"], -val_m["loss"])
        if cur_score > best_score:
            best_score = cur_score
            best_epoch = epoch
            best_metrics = dict(val_m)
            no_improve = 0
            ckpt_obj = {
                "epoch": best_epoch,
                "model_state_dict": model.state_dict(),
                "val_metrics": best_metrics,
                "class_to_id": class_to_id,
                "id_to_class": id_to_class,
                "config": {"num_frames": 32, "image_size": 224, **model_cfg},
            }
            if is_multimodal and vocab_tensor is not None:
                ckpt_obj["vocab_text_embeddings"] = vocab_tensor.detach().cpu()
            torch.save(ckpt_obj, run_dir / "best_model.pt")
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    elapsed = round(time.time() - t0, 2)
    pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)
    (run_dir / "val_metrics.json").write_text(json.dumps(best_metrics, indent=2), encoding="utf-8")
    r5_str = f" | V2T_R@5={best_metrics['v2t_recall@5']*100:5.2f}%" if is_multimodal else ""
    print(
        f"[{exp_id:28s}] best_ep={best_epoch:2d} | val_loss={best_metrics['loss']:.3f} | "
        f"val_top1={best_metrics['top1_accuracy']*100:5.2f}% | val_top5={best_metrics['top5_accuracy']*100:5.2f}% | "
        f"f1={best_metrics['macro_f1']:.4f}{r5_str} | uniq={best_metrics['unique_predicted_classes']:3d}/100 | {elapsed}s"
    )
    return {"experiment_id": exp_id, "best_epoch": best_epoch, "history": history, "val_metrics": best_metrics, "ckpt_path": str(run_dir / "best_model.pt")}

## 6. Run Regularized Visual v2 + Multimodal Fusion Ablations


In [ ]:
# ==============================================================================
# RUN VISUAL V2 & MULTIMODAL ABLATION SUITE (VALIDATION SPLIT)
# ==============================================================================
runs = []

# 1. Visual v2 (Regularized 128-D Transformer + Cutout + Mixup + Weight Decay 1e-2)
r_vis2 = train_and_eval_run(
    "visual_transformer_reg",
    VisualClassifier(NUM_CLASSES, feature_dim=FEATURE_DIM, hidden_dim=128, num_layers=2, dropout=0.45, temporal_type="transformer", pooling="attention"),
    is_multimodal=False,
    model_cfg={"feature_dim": FEATURE_DIM, "hidden_size": 128, "num_layers": 2, "dropout": 0.45, "temporal": "transformer", "bidirectional": False, "pooling": "attention"},
)
runs.append((r_vis2, "VisualTransformer(128,reg)", "visual_only", "none"))

# 2. Multimodal Contrastive (Raw Gloss Embeddings)
r_con_gloss = train_and_eval_run(
    "mm_contrastive_gloss",
    MultimodalASLModel(NUM_CLASSES, visual_feature_dim=FEATURE_DIM, temporal_hidden_dim=128, temporal_layers=2, temporal_type="transformer", pooling="attention", text_dim=384, embed_dim=128, dropout=0.45, fusion_mode="contrastive"),
    is_multimodal=True,
    vocab_tensor=vocab_gloss,
    model_cfg={"feature_dim": FEATURE_DIM, "hidden_size": 128, "num_layers": 2, "embed_dim": 128, "dropout": 0.45, "temporal": "transformer", "bidirectional": False, "pooling": "attention", "fusion_mode": "contrastive"},
)
runs.append((r_con_gloss, "MM-Transformer(gloss)", "multimodal", "contrastive"))

# 3. Multimodal Contrastive (ASL Prompt Embeddings)
r_con_prompt = train_and_eval_run(
    "mm_contrastive_prompt",
    MultimodalASLModel(NUM_CLASSES, visual_feature_dim=FEATURE_DIM, temporal_hidden_dim=128, temporal_layers=2, temporal_type="transformer", pooling="attention", text_dim=384, embed_dim=128, dropout=0.45, fusion_mode="contrastive"),
    is_multimodal=True,
    vocab_tensor=vocab_prompt,
    model_cfg={"feature_dim": FEATURE_DIM, "hidden_size": 128, "num_layers": 2, "embed_dim": 128, "dropout": 0.45, "temporal": "transformer", "bidirectional": False, "pooling": "attention", "fusion_mode": "contrastive"},
)
runs.append((r_con_prompt, "MM-Transformer(prompt)", "multimodal", "contrastive"))

# Select stronger text embedding representation based strictly on validation Top-1 (tie-break V2T MRR)
use_gloss = (
    (r_con_gloss["val_metrics"]["top1_accuracy"], r_con_gloss["val_metrics"]["v2t_mrr"])
    >= (r_con_prompt["val_metrics"]["top1_accuracy"], r_con_prompt["val_metrics"]["v2t_mrr"])
)
best_vocab = vocab_gloss if use_gloss else vocab_prompt
best_vocab_name = "gloss" if use_gloss else "prompt"
print(f"\nSelected stronger text variant on val: {best_vocab_name}")

# 4. Multimodal Hybrid Fusion (Transformer)
r_hybrid = train_and_eval_run(
    "mm_hybrid_transformer",
    MultimodalASLModel(NUM_CLASSES, visual_feature_dim=FEATURE_DIM, temporal_hidden_dim=128, temporal_layers=2, temporal_type="transformer", pooling="attention", text_dim=384, embed_dim=128, dropout=0.45, fusion_mode="hybrid", fusion_alpha=0.5),
    is_multimodal=True,
    vocab_tensor=best_vocab,
    model_cfg={"feature_dim": FEATURE_DIM, "hidden_size": 128, "num_layers": 2, "embed_dim": 128, "dropout": 0.45, "temporal": "transformer", "bidirectional": False, "pooling": "attention", "fusion_mode": "hybrid", "fusion_alpha": 0.5},
)
runs.append((r_hybrid, f"MM-Hybrid({best_vocab_name})", "multimodal", "hybrid"))

# 5. Multimodal Gated Fusion (Transformer)
r_gated = train_and_eval_run(
    "mm_gated_fusion_transformer",
    MultimodalASLModel(NUM_CLASSES, visual_feature_dim=FEATURE_DIM, temporal_hidden_dim=128, temporal_layers=2, temporal_type="transformer", pooling="attention", text_dim=384, embed_dim=128, dropout=0.45, fusion_mode="gated_fusion", fusion_alpha=0.5),
    is_multimodal=True,
    vocab_tensor=best_vocab,
    model_cfg={"feature_dim": FEATURE_DIM, "hidden_size": 128, "num_layers": 2, "embed_dim": 128, "dropout": 0.45, "temporal": "transformer", "bidirectional": False, "pooling": "attention", "fusion_mode": "gated_fusion", "fusion_alpha": 0.5},
)
runs.append((r_gated, f"MM-GatedFusion-Trans({best_vocab_name})", "multimodal", "gated_fusion"))

# 6. Multimodal Gated Fusion (BiLSTM)
r_gated_lstm = train_and_eval_run(
    "mm_gated_fusion_bilstm",
    MultimodalASLModel(NUM_CLASSES, visual_feature_dim=FEATURE_DIM, temporal_hidden_dim=128, temporal_layers=2, temporal_type="lstm", bidirectional=True, pooling="attention", text_dim=384, embed_dim=128, dropout=0.45, fusion_mode="gated_fusion", fusion_alpha=0.5),
    is_multimodal=True,
    vocab_tensor=best_vocab,
    model_cfg={"feature_dim": FEATURE_DIM, "hidden_size": 128, "num_layers": 2, "embed_dim": 128, "dropout": 0.45, "temporal": "lstm", "bidirectional": True, "pooling": "attention", "fusion_mode": "gated_fusion", "fusion_alpha": 0.5},
)
runs.append((r_gated_lstm, f"MM-GatedFusion-BiLSTM({best_vocab_name})", "multimodal", "gated_fusion"))

## 7. Validation Curves, Results Tracker & Summary Output


In [ ]:
# ==============================================================================
# PLOT VALIDATION CURVES, UPDATE RESULTS TRACKER & PRINT SUMMARY
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for res, lbl, _, _ in runs:
    hdf = pd.DataFrame(res["history"])
    axes[0].plot(hdf["epoch"], hdf["val_loss"], label=lbl)
    axes[1].plot(hdf["epoch"], hdf["val_top1"] * 100.0, label=lbl)
    axes[2].plot(hdf["epoch"], hdf["val_top5"] * 100.0, label=lbl)

axes[0].set_title("Validation Loss (Regularized + Multimodal)")
axes[1].set_title("Validation Top-1 Accuracy (%)")
axes[2].set_title("Validation Top-5 Accuracy (%)")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)
plt.tight_layout()
plot_path = PLOTS_DIR / "04_multimodal_training_curves.png"
plt.savefig(plot_path, dpi=130)
plt.show()

summary_rows = []
for res, mdl_name, mod_name, fus_name in runs:
    vm = res["val_metrics"]
    summary_rows.append({
        "experiment_id": res["experiment_id"],
        "model": mdl_name,
        "modality": mod_name,
        "fusion_mode": fus_name,
        "split": "val",
        "top1_accuracy": round(vm["top1_accuracy"], 4),
        "top5_accuracy": round(vm["top5_accuracy"], 4),
        "macro_f1": round(vm["macro_f1"], 4),
        "v2t_recall@1": round(vm["v2t_recall@1"], 4) if "v2t_recall@1" in vm else np.nan,
        "v2t_recall@5": round(vm["v2t_recall@5"], 4) if "v2t_recall@5" in vm else np.nan,
        "t2v_recall@1": round(vm["t2v_recall@1"], 4) if "t2v_recall@1" in vm else np.nan,
        "mrr": round(vm["v2t_mrr"], 4) if "v2t_mrr" in vm else np.nan,
        "checkpoint_path": res["ckpt_path"],
    })

new_df = pd.DataFrame(summary_rows)
res_csv = EXP_DIR / "results_summary.csv"
if res_csv.is_file():
    old_df = pd.read_csv(res_csv)
    old_df = old_df[~((old_df["experiment_id"].isin(new_df["experiment_id"])) & (old_df["split"] == "val"))]
    frames = [df.dropna(axis=1, how="all") for df in (old_df, new_df) if not df.empty]
    merged_df = pd.concat(frames, ignore_index=True)
else:
    merged_df = new_df
merged_df.to_csv(res_csv, index=False)

print("=" * 72)
print("NOTEBOOK 04 (MULTIMODAL TRAINING ON VALIDATION) COMPLETED")
print("=" * 72)
cols = ["experiment_id", "fusion_mode", "top1_accuracy", "top5_accuracy", "macro_f1", "v2t_recall@1", "v2t_recall@5", "mrr"]
print(new_df[cols].to_string(index=False))
print(f"\nText variant selected for fusion runs: {best_vocab_name}")
print(f"Saved plot   : {plot_path}")
print(f"Results CSV  : {res_csv}")